# Send Email from Fabric Notebook via On-Prem SMTP

Sends an HTML email using an internal SMTP relay with a service account.

**Prerequisites**
- SMTP host reachable from Fabric (public internet egress).
- Service account credentials stored in Azure Key Vault.
- Service account has *send / relay* permission on the SMTP server.
- If the SMTP server uses an internal CA, the CA root cert must be available to the notebook (uploaded to a Lakehouse File or attached as a resource).

## 1. Configuration
Update these values for your environment.

In [ ]:
SMTP_SERVER = "smtp.yourcompany.com"      # host or smart-host FQDN
SMTP_PORT   = 587                           # 587 = STARTTLS, 465 = implicit SSL, 25 = plain
USERNAME    = "svc_fabric@yourcompany.com"

KEYVAULT_URL = "https://<your-vault>.vault.azure.net/"
SECRET_NAME  = "smtp-password"

SENDER     = USERNAME
RECIPIENTS = ["recipient@yourcompany.com"]
SUBJECT    = "HTML Email from Fabric Notebook"

# Set to a path of an internal CA bundle (.pem) if the SMTP server uses an internal CA.
# Leave as None to use Python's default trust store (public CAs).
CA_BUNDLE_PATH = None  # e.g. "/lakehouse/default/Files/certs/internal-ca.pem"

## 2. Retrieve password from Key Vault

In [ ]:
import notebookutils

PASSWORD = notebookutils.credentials.getSecret(KEYVAULT_URL, SECRET_NAME)
print("Password retrieved (length):", len(PASSWORD))

## 3. Build the email message

In [ ]:
from email.mime.text import MIMEText

html_body = """\
<html>
  <body>
    <h2>Hello,</h2>
    <p>This is an <b>HTML</b> email sent from a <a href=\"https://learn.microsoft.com/fabric\">Microsoft Fabric</a> notebook.</p>
  </body>
</html>
"""

msg = MIMEText(html_body, "html")
msg["Subject"] = SUBJECT
msg["From"]    = SENDER
msg["To"]      = ", ".join(RECIPIENTS)

## 4. Send via SMTP (STARTTLS)

In [ ]:
import smtplib, ssl

if CA_BUNDLE_PATH:
    context = ssl.create_default_context(cafile=CA_BUNDLE_PATH)
else:
    context = ssl.create_default_context()

try:
    with smtplib.SMTP(SMTP_SERVER, SMTP_PORT, timeout=30) as server:
        server.ehlo()
        server.starttls(context=context)
        server.ehlo()
        server.login(USERNAME, PASSWORD)
        server.send_message(msg)
    print("Email sent successfully.")
except smtplib.SMTPAuthenticationError as e:
    print(f"Authentication failed: {e}")
    raise
except smtplib.SMTPException as e:
    print(f"SMTP error: {e}")
    raise
except OSError as e:
    print(f"Network/TLS error: {e}")
    raise

## 5. (Optional) Implicit SSL on port 465
Use this cell *instead of* the previous one if your server requires implicit SSL.

In [ ]:
# import smtplib, ssl
# context = ssl.create_default_context(cafile=CA_BUNDLE_PATH) if CA_BUNDLE_PATH else ssl.create_default_context()
# with smtplib.SMTP_SSL(SMTP_SERVER, 465, context=context, timeout=30) as server:
#     server.login(USERNAME, PASSWORD)
#     server.send_message(msg)
# print("Email sent successfully (SSL).")

In [ ]:
from azure.identity import ClientSecretCredential
from azure.keyvault.secrets import SecretClient

vault_url = f"https://{azure_key_vault_name}.vault.azure.net"

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret,
)
client = SecretClient(vault_url=vault_url, credential=credential)

for s in client.list_properties_of_secrets():
    print(s.name)